# Part 2 - Customer Service Multi-Agent System with CrewAI

*Last updated: November 2026*

A two-level customer support crew:
- **Level 1 - Customer Care Specialist** answers the customer's question, using a **web-scraping tool**
  to read the CrewAI documentation.
- **Level 2 - Support Quality Analyst** reviews the answer for accuracy and completeness.

You will also see **tools assigned to a task** and **crew memory** (`memory=True`).

## How to run this notebook

1. Set up the course environment once, in the course folder: `uv sync`.
   New to coding? Follow `BEGINNER_SETUP.md` step by step. Otherwise see `LOCAL_SETUP.MD`.
2. Select the **`.venv`** Python kernel (*Python 3.12*) for this notebook.
3. Copy `.env.example` (course folder) to `.env` and add your keys. The notebook finds `.env` automatically.

**Keys and settings used in this notebook**
- `OPENAI_API_KEY` (required)

Run the cells from top to bottom.

> **Compatibility update (differs from the recording)**
> - The recorded CrewAI 0.28 can no longer be installed: its tools package depends on `lancedb` versions that were removed from PyPI.
>   This notebook uses **CrewAI 1.x**. Agents, tasks, crews and `{placeholders}` work the same way.
> - `verbose=2` is now `verbose=True`, and the console output looks different from the video.
> - The scraping tool is still `ScrapeWebsiteTool` from `crewai_tools` (the `config={}` argument is no longer used).
> - No Serper key is needed: the Serper search tool was imported in the recording but never used.
> - The crew uses CrewAI's default OpenAI model (`gpt-4.1-mini`). `memory=True` also makes OpenAI embedding calls and stores memory files on your computer.

In [ ]:
# Setup check: run this cell first.
# Nothing is installed here. All packages were installed once with `uv sync` (see LOCAL_SETUP.MD).
import sys
from pathlib import Path

python_version = sys.version.split()[0]

if sys.version_info[:2] != (3, 12):
    print(f"Warning: this course is tested on Python 3.12, but this kernel runs Python {python_version}.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
elif Path(sys.prefix).name != ".venv":
    print("Warning: this kernel is not the course .venv environment, so some packages may be missing.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
else:
    print("Ready: course .venv kernel | Python", python_version)

In [ ]:
# Load API keys from the .env file, so keys are never written in the notebook.
# load_dotenv() looks for .env in this folder and then in each parent folder (the course folder).
import os

from dotenv import load_dotenv

load_dotenv()

REQUIRED_KEYS = ['OPENAI_API_KEY']

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    print("Missing required keys:", ", ".join(missing), "- see 'How to run this notebook' above.")
else:
    print("All required API keys are loaded.")

In [ ]:
# Hide library deprecation warnings so the lesson output stays readable
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from crewai import Agent, Task, Crew

In [ ]:
# Define first Agent (Level 1)
customer_assist_agent = Agent(
    role="Customer Care Specialist",
    goal="Deliver the most courteous and effective "
         "customer support on your team.",
    backstory=(
        "You are a valued team member at Crew AI (https://docs.crewai.com/introduction) "
        "and currently assigned to assist {customer}, a crucial client "
        "for the company."
        "Your mission is to ensure the highest quality of support!"
        "Provide detailed and thorough responses, "
        "avoiding any assumptions."
    ),
    allow_delegation=False,
    verbose=True
)

In [ ]:
# Define second agent (Level 2)
quality_check_agent = Agent(
    role="Support Quality Analyst",
    goal="Be recognized for ensuring the highest "
         "standards of support quality in your team.",
    backstory=(
        "You are a key contributor at CrewAI (https://docs.crewai.com/introduction), "
        "working alongside your team to review a support request from {customer}. "
        "Your responsibility is to ensure that the Customer Care Specialist "
        "delivers top-notch service.\n"
        "It's your duty to confirm that all responses are thorough, "
        "accurate, and free of assumptions."
    ),
    verbose=True
)

In [ ]:
from crewai_tools import ScrapeWebsiteTool

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/introduction"
)

In [ ]:
customer_query_task = Task(
    description=(
        "{customer} has submitted a critical request:\n"
        "{inquiry}\n\n"
        "{person} from {customer} initiated the inquiry. "
        "Leverage all available resources and knowledge to "
        "deliver outstanding support."
        "Your goal is to provide a thorough and accurate resolution "
        "to the customer's question."
    ),
    expected_output=(
        "A comprehensive and well-researched response to the customer's inquiry "
        "that addresses every aspect of their concern.\n"
        "The response must reference all the tools and information sources used, "
        "including external materials or solutions. "
        "Ensure that the answer is exhaustive, leaves no room for follow-up questions, "
        "and maintains a professional, approachable tone."
    ),
    tools=[docs_scrape_tool],
    agent=customer_assist_agent,
)

In [ ]:
response_review_task = Task(
    description=(
        "Evaluate the response prepared by the Customer Care Specialist for {customer}'s inquiry. "
        "Ensure the reply meets the highest standards of accuracy, completeness, and clarity "
        "expected in customer service.\n"
        "Confirm that every aspect of the customer's question has been addressed "
        "in a friendly and approachable manner.\n"
        "Verify the inclusion of appropriate references and sources used "
        "to gather the information, ensuring the response is well-supported and "
        "leaves no loose ends."
    ),
    expected_output=(
        "A polished, detailed, and ready-to-send response that completely addresses "
        "the customer's inquiry.\n"
        "The response should reflect all necessary feedback and adjustments while "
        "maintaining a relaxed yet professional tone that aligns with our cool and casual company culture."
    ),
    agent=quality_check_agent,
)

In [ ]:
crewCSMAS = Crew(
    agents=[customer_assist_agent, quality_check_agent],
    tasks=[customer_query_task, response_review_task],
    verbose=True,
    memory=True
)

In [ ]:
inputs = {
    "customer": "Unfold data science",
    "person": "Aman Kumar",
    "inquiry": "I need help with Crew AI "
               "and Understanding about Crew and Agent "
               "How to use Crew and Agent in CrewAI "
               "Can you provide guidance?"
}
result = crewCSMAS.kickoff(inputs=inputs)

In [ ]:
# Final, quality-checked reply to the customer
print(result.raw)